In [324]:
from abc import ABC, abstractmethod
import warnings
import inspect
import logging
import copy
import numpy as np
import scipy.stats
from scipy.stats import multivariate_normal
from scipy import stats
from collections.abc import Iterable

In [2]:
# Start with simple two node Gaussian case
# Outer (macro) graph should have standard adjacency matrix
# Each edge of the macro graph also has a corresponding bottleneck function (this could be saved in the same object as the adjacency matrix)
# Each vertex of the macro graph has some internal structure: either singleton (then it's just a "regular" node), or a chain component

In [ ]:
# TODO:
# - Define different sampling schemes that can be internally used as mechanism_micro

In [37]:
rng = np.random.default_rng()
n = 4
P = rng.random(size=(n, n))
P = P @ P.T # Make this thing pd
M = np.ones((n, n)) # mask to get the correct structure for a Gaussian chain component
M[0,3] = M[1,2] = M[2,1] = M[3,0] = 0
P = np.where(M, P, 0) + np.identity(n)

In [38]:
np.linalg.eigvals(P)

array([3.50753778, 0.73634218, 2.19790003, 1.58606186])

In [39]:
P

array([[2.20416897, 0.53039113, 0.32240517, 0.        ],
       [0.53039113, 2.40702632, 0.        , 1.0144814 ],
       [0.32240517, 0.        , 1.26209464, 0.48754472],
       [0.        , 1.0144814 , 0.48754472, 2.15455192]])

In [40]:
# Define some toy macrovariables
m1 = MacroVar(parents=None, bottleneck_fcts=None, n_micro=4, mechanism_micro=multivariate_normal(cov=stats.Covariance.from_precision(P)))

In [ ]:
# Preprocessing: get adjacency matrix, get bottleneck variable functions

In [ ]:
# Sampling loop
# Input: scm, (nr. of samples?)
# Outer loop over macrovariables
# Get bottleneck variables
# Inner loop over microvariables

In [13]:
# Define a multivariate Gaussian that is a valid joint distr. of a MRF (Langevin)
# Sample a precision matrix that fulfills the necessary criteria
rng = np.random.default_rng()
n = 4 # nr of nodes
P = rng.random(size=(n, n)) # sample dense n x n matrix
P = P @ P.T # make this object symmetric
M = np.ones((n, n)) # this will become the mask for the correct adjacency structure
M[0,3] = M[1,2] = M[2,1] = M[3,0] = 0 # hardcoding sparsity for now
P += np.identity(n) # make positive definite
C = -0.5 * P
C = np.where(M, C, 0) # apply sparsity map

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [5]:
# Check that this is in fact negative definite, reject if not
np.linalg.eigvals(C)

array([-2.365156  , -0.13551986, -1.11162801, -1.26851452])

In [6]:
P = -0.5 * C

In [7]:
P

array([[ 0.6800688 ,  0.33431444,  0.2990518 , -0.        ],
       [ 0.33431444,  0.64752164, -0.        ,  0.26057566],
       [ 0.2990518 , -0.        ,  0.60144012,  0.18852302],
       [-0.        ,  0.26057566,  0.18852302,  0.51137864]])

In [49]:
bla = np.linalg.inv(P)

In [50]:
bla

array([[ 3.0332636 , -2.09261674, -1.66158577,  1.28254048],
       [-2.09261674,  4.17252699,  1.66655491, -1.94432167],
       [-1.66158577,  1.66655491,  3.3887231 , -1.79033726],
       [ 1.28254048, -1.94432167, -1.79033726,  2.75761678]])

In [8]:
# Define the observational joint distribution
obs_dist = multivariate_normal(cov=stats.Covariance.from_precision(P))

In [52]:
# Draw an observational sample
obs_sample = obs_dist.rvs(size=10)

In [53]:
obs_sample

array([[ 3.1526128 , -2.41797563, -1.86224375,  0.85823009],
       [-1.57020501,  1.14068378,  2.14960103, -1.35617374],
       [ 1.44958651, -1.98012128, -1.36053033,  0.79046988],
       [ 0.12058689, -1.23859573, -0.98360359,  2.6844182 ],
       [ 0.77418757,  1.14223129,  0.90447639, -1.6290778 ],
       [-0.62206515,  2.84150214, -3.29256105,  0.37403547],
       [ 2.11858069,  0.70454999, -0.89880943,  1.43041558],
       [-1.48835131,  1.25146183,  0.70517533, -0.53349772],
       [-0.80609975,  1.0861691 ,  0.23029518, -0.20156469],
       [ 2.04233704, -0.64411858, -0.11521497,  0.61332297]])

In [66]:
# Draw an interventional sample
iv_target = 2 # idx of intervention target
iv_value = [2,3]
# Get components of joint covariance
cov = np.linalg.inv(P)


In [79]:
cov

array([[ 3.0332636 , -2.09261674, -1.66158577,  1.28254048],
       [-2.09261674,  4.17252699,  1.66655491, -1.94432167],
       [-1.66158577,  1.66655491,  3.3887231 , -1.79033726],
       [ 1.28254048, -1.94432167, -1.79033726,  2.75761678]])

In [80]:
E_11 = np.delete(np.delete(cov, iv_target, axis=0), iv_target, axis=1) # E_11

In [81]:
if isinstance(iv_target, Iterable):
    E_22 = cov[iv_target, :][:, iv_target] # E_22
else:
    E_22 = cov[iv_target, iv_target]

In [82]:
E_12 = np.delete(cov, iv_target, axis=0)[:, iv_target] # E_12

In [83]:
print(E_11)
print(E_22)
print(E_12)

[[ 3.0332636  -2.09261674  1.28254048]
 [-2.09261674  4.17252699 -1.94432167]
 [ 1.28254048 -1.94432167  2.75761678]]
3.3887230985568553
[-1.66158577  1.66655491 -1.79033726]


In [ ]:
mean_iv = 0 # TODO
cov_iv = 0 # TODO

In [3]:
# Define bottleneck function
# For now, let's just take the mean of the inputs
f_y = lambda x: np.mean(x, axis=1)

In [12]:
# Sample from the observational distribution
x_1_obs = obs_dist.rvs(size=100)

In [13]:
x_1_obs.shape

(100, 4)

In [21]:
# Push forward through bottleneck function
y_1 = f_y(x_1_obs)

In [22]:
y_1.shape

(100,)

In [23]:
y_1

array([-4.96287449e-01,  3.38398246e-02, -4.02112461e-01, -4.45280564e-01,
       -2.15142160e-01,  1.76603783e-01, -3.74469779e-01, -2.04557946e-01,
       -1.52260930e-01, -4.30317542e-02, -2.02269470e-01, -2.20787179e-01,
       -9.64943587e-02, -4.00217175e-02,  6.21396706e-01, -7.98015068e-02,
        9.22393507e-02, -4.51617130e-01,  4.76490192e-02,  8.90415966e-01,
        9.38378041e-02,  1.91451986e-01, -1.06981370e-01, -2.28121320e-01,
        4.98371064e-01, -3.30465391e-02, -6.10579781e-01,  5.38848558e-01,
       -1.16884591e-01,  3.73599780e-01, -5.98153482e-01,  3.47261129e-01,
       -3.64892091e-01,  1.52168533e-01,  3.05841420e-01, -7.91371024e-01,
        1.08385432e-01, -2.30151713e-01,  2.31101403e-01, -2.90688742e-01,
       -7.04005850e-01, -9.88493980e-01, -5.72444710e-01, -9.11052399e-02,
        4.12016040e-01, -1.52452701e-01,  7.85220764e-02, -1.34547032e-01,
        2.46453475e-01,  8.16940691e-01,  2.78233122e-02, -1.63848930e-01,
       -8.36122234e-01,  

In [31]:
# Define equations for X2
# we still need to draw this guy from some kind of conditional Gaussian, or else our nice Langevin sampling intervention equivalence breaks
# For now, the effect of the bottleneck is only present in the mean:
mu_2 = 5 * np.mean(y_1) * np.ones(4)
# Sample covariance matrix
P_2 = sample_mrf_prec(dim=4, M=M)


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [14]:
# Draw an observational sample from the joint distribution
# X_1
mu_1 = 0
P_1 = sample_mrf_prec(dim=4, M=M, seed=1)
X_1 = multivariate_normal(mean=mu_1, cov=stats.Covariance.from_precision(P_1))
x_1_obs = X_1.rvs(size=100)

# Y_1
y_1_obs = f_y(x_1_obs)

# X_2
mu_2 = 5 * np.mean(y_1_obs) * np.ones(4)
P_2 = sample_mrf_prec(dim=4, M=M)
X_2 = multivariate_normal(mean=mu_2, cov=stats.Covariance.from_precision(P_2))
x_2_obs = X_2.rvs(size=100)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [80]:
# Draw an interventional sample
# define target and value (also have to specificy which macro node we target (two level indexing?))
iv_target = [0,1]
iv_value = [3,5]
# define function to automatically get the adapted covariance (wrapper for what is done above)
mu_1_bar, E_1_bar = get_cond_mean_cov(iv_target, iv_value, X_1.mean, X_1.cov)

X_1_bar = multivariate_normal(mean=mu_1_bar, cov=E_1_bar)
x_1_iv = X_1_bar.rvs(size=100)
# insert the iv value back into sample
# # think about how bottleneck value is included...(i.e. how are mean or cov modulated, which functional form)

In [465]:
def sample_mrf_prec(dim, M, rs):
    """
    Args:
        dim: int
            number of dimensions
        M: np.array
            array that encodes the sparsity structure of the precision matrix
            s.t. this is a valid MRF
        seed: int
            the random seed
    Returns:
        P: np.array
            sampled precision matrix
    """
    def sample():
        P = rs.random(size=(dim, dim)) # sample dense dim x dim matrix
        P = P @ P.T # make this object symmetric
        P += np.identity(dim) # make positive definite
        P = np.where(M, P, 0) # apply sparsity map
        return P

    P = sample()
    # Rejection sampling until P is pd
    while any(np.linalg.eigvals(P) < 0):
        P = sample()

    return P

In [328]:
def make_iterable(x):
    if isinstance(x, Iterable):
        return x
    else:
        return [x]

In [329]:
def get_cond_submats(target, E):
    """
    Get the submatrices of the covariance for the Gaussian conditioning formula.

    Args:
        target: list or int
            indices of intervention targets
        E: np.array
            original covariance matrix
    Returns:
        E_11, E_22, E_12: np.array
            submatrices of E
    """
    E_11 = np.delete(np.delete(E, target, axis=0), target, axis=1)

    target = make_iterable(target)

    E_22 = E[target, :][:, target]

    E_12 = np.delete(E, target, axis=0)[:, target]

    return E_11, E_22, E_12

In [438]:
def get_cond_mean_cov(iv_target, iv_value, E):
    """

    :param target:
    :param value:
    :param mu:
    :param E:
    :return:
    """
    iv_value = make_iterable(iv_value)

    E_11, E_22, E_12 = get_cond_submats(iv_target, E)

    if isinstance(E_22, np.ndarray):
        E_22_inv = np.linalg.inv(E_22)
    else: # if this is a scalar
        E_22_inv = [1/E_22]

    # mu_1 = np.delete(mu, iv_target)
    mu_bar = E_12 @ E_22_inv @ iv_value # treat distr. as zero mean Gaussian, original mean is added back later

    E_bar = E_11 - E_12 @ E_22_inv @ E_12.T

    return mu_bar, E_bar

In [421]:
class MacroCausalVar(object):
    """
    Only use jointly Gaussian version for now.
    """
    def __init__(self, parents, bottleneck_fcts, mechanism, n):
        assert ((isinstance(parents, Iterable) and
                 all(isinstance(parent, MacroCausalVar) for parent in parents)) or
                parents is None), "Parents of a variable must be list of MacroCausalVar objects, or None!"
        self.parents = parents

        assert ((isinstance(bottleneck_fcts, Iterable) and all(inspect.isfunction(bottleneck_fct) for bottleneck_fct in bottleneck_fcts)) or bottleneck_fcts is None), "Bottleneck functions must be a list of functions, or None!)"
        self.bottleneck_fcts = bottleneck_fcts # surjective functions (one for each parent) that map micro states to some summary statistic(s)

        assert isinstance(mechanism, SCBMMechanism), "Mechanism must be SCBMMechanism type!"
        self.mechanism = mechanism # Some object that we can sample from, that returns n_micro dimensional samples. Can also take arguments from the respective bottleneck vars

        self.n = n # number of internal (scalar) variables

        # Assigned during sampling
        self.value = None


In [435]:
class SCBM(object):
    """
    Structural Causal Bottleneck Model
    """
    def __init__(self, variables, seed=None):
        # Random seed for reproducibility
        self.seed = seed
        self.rs = np.random.RandomState(seed=self.seed)

        self.variables = variables # variables must be defined in correct causal ordering
        assert all(isinstance(var, MacroCausalVar) for var in self.variables), ("All SCBM variables must be MacroCausalVar objects!")

        self.intervention_flag = False

    def sample(self, size):
        """
        Draw random samples.
        :param size:
        Returns:
            values: list[np.array[size, n_microvars]] with len=n_variables
        """
        if self.intervention_flag:
            logging.warning("The SCM from which you are sampling has been intervened upon!")

        values = []

        # Loop over variables
        for i, var in enumerate(self.variables):
            # Currently, this sampling works for the langevin gaussian case, might have to adapt it later on
            # Sample from independent Gaussian
            noise = self.rs.multivariate_normal(mean=np.zeros(var.n), cov=np.eye(var.n), size=size)
            if var.parents is not None:
                # Get bottleneck values
                bottleneck_values = [bottleneck_fct(parent.value) for parent, bottleneck_fct in zip(var.parents, var.bottleneck_fcts)]

                value = var.mechanism(noise, *bottleneck_values)
            else: # Leaf nodes
                value = var.mechanism(noise)

            var.value = value

            values.append(value)

        return values

    def intervene(self, iv):
        self.intervention_flag = True

        # Loop over interventions
        for macro_target, target, value in zip(iv.macro_targets, iv.micro_targets, iv.values):
            # Reset sampled values to None
            self.variables[macro_target].value = None
            # Set mechanism
            self.variables[macro_target].mechanism.intervene(target, value)

    def intervent_sample(self, iv, size):
        # Make copy of SCM to not permanently alter state
        SCBM_copy = copy.deepcopy(self)
        # Intervene
        SCBM_copy.intervene(iv)
        # Sample
        return SCBM_copy.sample(size)


In [423]:
class Intervention(object):
    """
    Intervention class to intervene on SCBMs.
    Only supports hard interventions for now.
    """
    def __init__(self, macro_targets, micro_targets, values):
        # Make all arguments iterable if they are single elements
        macro_targets = make_iterable(macro_targets)
        if not any(hasattr(elem, '__len__') for elem in micro_targets): # if it is just a list of ints, not a list of lists
            micro_targets = [micro_targets]
        if not any(hasattr(elem, '__len__') for elem in values):
            values = [values]

        assert all(len(micro_target) == len(value) for micro_target, value in zip(micro_targets, values)), "Length of all micro_targets and values must match!"

        self.macro_targets = macro_targets
        self.micro_targets = micro_targets
        self.values = values


In [424]:
class SCBMMechanism(ABC):
    @abstractmethod
    def __init__(self):
        self.mechanism = self._get_mechanism()
        return NotImplementedError

    @abstractmethod
    def _get_mechanism(self, *args, **kwargs):
        raise NotImplementedError

    def __call__(self, *args, **kwargs):
        return self.mechanism(*args, **kwargs)

    # This method will modify the mechanism property
    @abstractmethod
    def intervene(self, target, value):
        raise NotImplementedError

In [444]:
class GaussianLangevinMechanism(SCBMMechanism):
    def __init__(self, mu, E):
        """

        :param mu:
        :param E:
        """
        if not inspect.isfunction(mu):
            mu_f = lambda : mu
            self.mu = mu_f
        else:
            self.mu = mu
        self.E = E

        super().__init__()

    def _get_mechanism(self):
        # Sample from a multivariate Gaussian
        L = np.linalg.cholesky(self.E)
        def f(noise, *args):
            return self.mu(*args) + (L @ noise.T).T
        return f

    def intervene(self, target, value):
        value = make_iterable(value)

        mu_bar, E_bar = get_cond_mean_cov(target, value, self.E)
        L_bar = np.linalg.cholesky(E_bar)

        def f(noise, *args):
            # Allocate array for output
            out = np.empty_like(noise)
            # Apply mu mechanism and drop intervened dims we don't need
            mu_mech = self.mu(*args)
            mu_mech = np.delete(mu_mech, target, axis=1)
            # Get correct size of noise
            noise = np.delete(noise, target, axis=1)
            sample = mu_mech + (mu_bar + (L_bar @ noise.T).T)
            # Populate out array at correct indices
            target_count = 0
            sample_count = 0
            for i in range(out.shape[1]):
                if i in target:
                    out[:, i] = value[target_count] * np.ones_like(out[:, i], dtype=float)
                    target_count += 1
                else:
                    out[:, i] = sample[:, sample_count]
                    sample_count += 1
            return out

        self.mechanism = f

In [411]:
np.linalg.inv(P_1)

array([[ 0.55938492, -0.39554744, -0.32728613,  0.27931103],
       [-0.39554744,  0.86470771,  0.31933959, -0.47865752],
       [-0.32728613,  0.31933959,  0.72143853, -0.37778306],
       [ 0.27931103, -0.47865752, -0.37778306,  0.82502635]])

In [412]:
test_mech = GaussianLangevinMechanism(mu=np.zeros(4), E=np.linalg.inv(P_1))

In [413]:
rng = np.random.default_rng()
noise = rng.multivariate_normal(mean=np.zeros(4), cov=np.eye(4), size=100).T

In [414]:
noise.shape

(4, 100)

In [54]:
test_out = test_mech(noise)

In [100]:
target = [1,2]
value = [0,1]

test_mech.intervene(target, value)
test_out_iv = test_mech(noise)

/var/folders/3t/tsy826vs2hqbz3y19mzxzp1w0000gn/T/ipykernel_5232/1637371373.py:21: UserWarning: Intervening!
  warnings.warn("Intervening!")


In [101]:
test_out_iv

array([[ 2.96863162e-01,  0.00000000e+00,  1.00000000e+00,
        -1.06863829e+00],
       [-7.69154503e-01,  0.00000000e+00,  1.00000000e+00,
         6.60398628e-01],
       [-1.13217583e+00,  0.00000000e+00,  1.00000000e+00,
         8.75990049e-01],
       [-7.96947303e-01,  0.00000000e+00,  1.00000000e+00,
        -1.34634088e+00],
       [-4.74065563e-01,  0.00000000e+00,  1.00000000e+00,
        -1.41366819e+00],
       [-8.15444180e-01,  0.00000000e+00,  1.00000000e+00,
         1.11292197e+00],
       [-5.72605473e-02,  0.00000000e+00,  1.00000000e+00,
        -1.41771971e+00],
       [-1.14006735e+00,  0.00000000e+00,  1.00000000e+00,
         6.02211180e-01],
       [-1.99925001e-03,  0.00000000e+00,  1.00000000e+00,
        -8.53225498e-01],
       [-6.18480541e-01,  0.00000000e+00,  1.00000000e+00,
        -2.50580418e+00],
       [-5.09921976e-01,  0.00000000e+00,  1.00000000e+00,
         6.55532433e-01],
       [-8.72111230e-01,  0.00000000e+00,  1.00000000e+00,
      

In [42]:
# Sparsity structure within macro nodes is determined by the structure of the precision matrix
# Use the same structure mask for both nodes in this example
n_micro = 4
M = np.ones((n_micro, n_micro)) # this will become the mask for the correct adjacency structure
M[0,3] = M[1,2] = M[2,1] = M[3,0] = 0 # hardcoding sparsity for now



In [466]:
# Let's define a simple two node SCBM
global_seed = 0
rs = np.random.RandomState(seed=global_seed)
# X1
n1 = 4
# We use the same sparsity structure within each node for simplicity. This is defined by the following binary mask:
M = np.ones((n1, n1))
M[0,3] = M[1,2] = M[2,1] = M[3,0] = 0 # hardcoding sparsity for now

P1 = sample_mrf_prec(dim=n1, M=M, rs=rs)
mech1 = GaussianLangevinMechanism(mu=np.zeros(n1), E=np.linalg.inv(P1))

X1 = MacroCausalVar(parents=None, bottleneck_fcts=None, mechanism=mech1, n=n1)

# X2
n2 = 4
f_y12 = lambda x: np.mean(x, axis=1)

P2 = sample_mrf_prec(dim=n2, M=M, rs=rs)
# Define function for mu2
mu2 = lambda x: [np.sin(x_elem) * np.ones(n2) for x_elem in x] # Need list comprehension to deal with multiple samples as single x input
mech1 = GaussianLangevinMechanism(mu=mu2, E=np.linalg.inv(P2))

X2 = MacroCausalVar(parents=[X1], bottleneck_fcts=[f_y12], mechanism=mech1, n=n2)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [467]:
test_scbm = SCBM(variables=[X1, X2], seed=0)

In [468]:
test_sample_obs = test_scbm.sample(size=100)

In [469]:
test_sample_obs[0].shape

(100, 4)

In [470]:
# Interventional sample
test_iv = Intervention(macro_targets=1, micro_targets=[2, 3], values=[2, 2])
test_sample_iv = test_scbm.intervent_sample(iv=test_iv, size=100)

In [471]:
test_sample_iv[0].shape

(100, 4)